# Splane&Scube tutorial (1/2): Identify uniform spatial domain on human brain MERFISH dataset

Dataset: 33 MERFISH slices of mouse brain ([here](https://zenodo.org/record/8167488))

## Data preprocessing

In [1]:
from SPACEL.setting import set_environ_seed
seed = 4
set_environ_seed(seed)
from SPACEL import Splane
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib

Setting environment seed: 4


In [ ]:
# load Merfish mouse brain 33 data
in_ = './Merfish_mouse/'
adata_list = []

for i in range(33):
    file_path = in_ + 'merfish_mouse_brain_slice' + str(i) + '.h5ad'
    ad = sc.read(file_path)
    ad.obs['original_clusters']=ad.obs['spa_cluster'].astype('category')
#     ad.obsm['spatial'] = ad.obsm['spatial'].to_numpy()[:, :2]
    adata_list.append(ad)

Here, we will incorporate the cell type composition predicted by **Spoint** into the spatial anndata object for subsequent spatial domain identification in **Splane** using the `add_cell_type_composition` function. This function takes a DataFrame containing the cell type composition matrix as input for spot-based spatial transcriptomic data or a series of cell type annotations as input for single-cell resolution spatial transcriptomic data.

In [ ]:
# Collect all unique cell types across all .h5ad files
all_cell_types = set()
for adata in adata_list:
    all_cell_types.update(adata.obs['original_clusters'].unique().tolist())

all_cell_types = sorted(list(all_cell_types)) 
# all_cell_types

In [ ]:
for i, adata in enumerate(adata_list):    
    # Add cell type composition
    Splane.utils.add_cell_type_composition(adata, celltype_anno=adata.obs['annotation'])
    # Add missing cell type columns as 0s
    for ct in all_cell_types:
        if ct not in adata.obs.columns:
            adata.obs[ct] = 0.0
            
    # Reorder columns consistently
    ordered_cols = [c for c in adata.obs.columns if c not in all_cell_types] + all_cell_types
    adata.obs = adata.obs[ordered_cols]
    adata.uns['celltypes'] = np.array(all_cell_types, dtype=object)

## Training Splane model

In this step, we initialize the Splane model by ``Splane.init_model(...)`` using the anndata object list as input. The ``n_clusters`` parameter determines the number of spatial domains to be identified. The ``k`` parameter controls the degree of neighbors considered in the model, with a larger ``k`` value resulting in more emphasis on global structure rather than local structure. The ``gnn_dropout`` parameter influences the level of smoothness in the model’s predictions, with a higher ``gnn_dropout`` value resulting in a smoother output that accommodates the sparsity of the spatial transcriptomics data. 

We train the model by ``splane.train(...)`` to obtain latent feature of each spots/cells. The parameter ``d_l`` affects the level of batch effect correction between slices. By default, ``d_l`` is ``0.2`` for spatial transcriptomics data with single cell resolution.

Then, we can identify the spatial domain to which each spot/cell belongs by ``splane.identify_spatial_domain(...)``. By default, the results will be saved in ``spatial_domain`` column in ``.obs``. If the key parameter is provided, the results will be saved in ``.obs[key]``.

In [5]:
splane_model = Splane.init_model(adata_list, n_clusters=7,use_gpu=True,n_neighbors=25, gnn_dropout=0.5,seed=seed)
splane_model.train(d_l=0.2)
splane_model.identify_spatial_domain()

Setting global seed: 4
Calculating cell type weights...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/utils.py:76: RuntimeWarning: invalid value encountered in double_scalars
  return len(x) / W * inum / z2ss


Generating GNN inputs...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:11: RuntimeWarning: divide by zero encountered in power
  d = sp.diags(np.power(np.array(adj.sum(1)), -1).flatten(), 0)


Calculating largest eigenvalue of normalized graph Laplacian...
Calculating Chebyshev polynomials up to order 2...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:57: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:641.)
  return torch.sparse.FloatTensor(indices, values, shape)


Setting the weight of similarity loss to 15.444


The best epoch 120 total loss=-16.345 g loss=-15.647 d loss=3.491 d acc=0.058 simi loss=-0.997 db loss=0.579:  58%|█████▊    | 175/300 [10:32<07:31,  3.61s/it]

Stop trainning because of loss convergence


In [ ]:
sc.concat(adata_list).write(f'./{seed}/merfish_mouse_brain.h5ad')